# Bài 4: Mẫu thiết kế (Design Patterns) cho Khoa học dữ liệu

## Mục tiêu bài học

Sau bài học này, bạn sẽ có thể:
- Hiểu khái niệm **Mẫu thiết kế (Design Pattern)** là gì và tại sao chúng hữu ích.
- Nhận biết và áp dụng một số mẫu thiết kế phổ biến và phù hợp với các bài toán khoa học dữ liệu.
- Sử dụng **Factory Pattern** để tạo các đối tượng phức tạp một cách linh hoạt.
- Sử dụng **Strategy Pattern** để đóng gói và hoán đổi các thuật toán.
- Hiểu ý tưởng của **Singleton Pattern** và các trường hợp sử dụng của nó.
- Viết code có cấu trúc, dễ mở rộng và dễ bảo trì hơn bằng cách áp dụng các mẫu đã được kiểm chứng.

## 1. Mẫu thiết kế (Design Patterns) là gì?

Trong quá trình phát triển phần mềm, các lập trình viên thường xuyên gặp phải những vấn đề lặp đi lặp lại. **Mẫu thiết kế** là những **giải pháp tổng quát, đã được kiểm chứng và có thể tái sử dụng** cho các vấn đề phổ biến đó trong một bối cảnh nhất định.

Hãy tưởng tượng nó giống như công thức nấu ăn:
- **Vấn đề:** Bạn muốn làm một chiếc bánh.
- **Giải pháp (Mẫu thiết kế):** Một công thức làm bánh (liệt kê nguyên liệu, các bước thực hiện).

Công thức không phải là chiếc bánh cuối cùng, nó chỉ là hướng dẫn. Bạn có thể thay đổi một vài nguyên liệu (ví dụ: dùng sô cô la đen thay vì sô cô la trắng) nhưng vẫn tuân theo các bước chính. Tương tự, mẫu thiết kế cung cấp một "bộ xương" cho giải pháp, và bạn sẽ tùy chỉnh nó cho phù hợp với bài toán cụ thể của mình.

**Tại sao chúng quan trọng?**
- **Ngôn ngữ chung:** Giúp các lập trình viên giao tiếp hiệu quả hơn. Thay vì giải thích một giải pháp phức tạp, bạn chỉ cần nói "Hãy dùng Strategy Pattern ở đây".
- **Giải pháp đã được kiểm chứng:** Bạn không cần phải "phát minh lại bánh xe". Các mẫu này đã được nhiều người sử dụng và chứng minh là hiệu quả.
- **Code tốt hơn:** Thúc đẩy việc viết code có cấu trúc, linh hoạt, dễ bảo trì và mở rộng.

## 2. Factory Pattern (Mẫu Nhà máy)

**Ý tưởng:** Cung cấp một "nhà máy" chuyên để "sản xuất" các đối tượng. Thay vì gọi trực tiếp constructor của một class (ví dụ `MyModel()`), bạn sẽ yêu cầu nhà máy tạo đối tượng cho bạn.

**Vấn đề cần giải quyết:** Khi bạn có nhiều class con và bạn muốn quyết định sẽ khởi tạo class con nào dựa trên một tham số (ví dụ: một chuỗi string). Việc sử dụng một chuỗi `if/elif/else` dài để khởi tạo đối tượng sẽ làm code của bạn trở nên cồng kềnh và khó mở rộng.

**Ví dụ trong Data Science:** Tạo các đối tượng tiền xử lý dữ liệu (preprocessor) dựa trên tên của chúng.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder

# --- Các class sản phẩm ---
class Preprocessor:
    """Class cha (interface) cho các bộ tiền xử lý."""
    def process(self, data):
        raise NotImplementedError

class Scaler(Preprocessor):
    def __init__(self, scaler_instance):
        self._scaler = scaler_instance
    def process(self, data):
        print(f"Áp dụng {self._scaler.__class__.__name__}...")
        return self._scaler.fit_transform(data)

class Encoder(Preprocessor):
    def __init__(self):
        self._encoder = OneHotEncoder()
    def process(self, data):
        print("Áp dụng OneHotEncoder...")
        return self._encoder.fit_transform(data)

# --- Nhà máy (Factory) ---
class PreprocessorFactory:
    def get_preprocessor(self, name: str) -> Preprocessor:
        """Phương thức nhà máy, trả về một object preprocessor dựa trên tên."""
        if name == 'standard_scaler':
            return Scaler(StandardScaler())
        elif name == 'min_max_scaler':
            return Scaler(MinMaxScaler())
        elif name == 'one_hot_encoder':
            return Encoder()
        else:
            raise ValueError(f"Preprocessor không xác định: {name}")

# --- Sử dụng nhà máy ---
factory = PreprocessorFactory()

# Thay vì phải tự khởi tạo StandardScaler, MinMaxScaler...
# chúng ta chỉ cần yêu cầu nhà máy
scaler1 = factory.get_preprocessor('standard_scaler')
scaler2 = factory.get_preprocessor('min_max_scaler')
encoder1 = factory.get_preprocessor('one_hot_encoder')

print(scaler1)
print(scaler2)
print(encoder1)

# Ưu điểm: Nếu muốn thêm một preprocessor mới, ví dụ 'robust_scaler',
# bạn chỉ cần thêm một class mới và một câu `elif` trong factory.
# Code sử dụng factory không cần thay đổi.

## 3. Strategy Pattern (Mẫu Chiến lược)

**Ý tưởng:** Định nghĩa một họ các thuật toán, đóng gói mỗi thuật toán vào một class riêng, và làm cho chúng có thể hoán đổi cho nhau. Class chính (context) sẽ chứa một tham chiếu đến một object "chiến lược" và ủy thác công việc cho nó.

**Vấn đề cần giải quyết:** Khi một class có nhiều hành vi khác nhau dựa trên một điều kiện nào đó. Thay vì dùng một câu `if/elif/else` lớn bên trong một phương thức, bạn tách các hành vi đó ra các class chiến lược riêng biệt.

**Ví dụ trong Data Science:** Xử lý giá trị thiếu (missing values) bằng các chiến lược khác nhau (điền giá trị trung bình, trung vị, hoặc một hằng số).

In [ ]:
import pandas as pd
import numpy as np
from abc import ABC, abstractmethod

# --- Interface và các class Chiến lược ---
class ImputationStrategy(ABC):
    """Interface chung cho các chiến lược điền dữ liệu."""
    @abstractmethod
    def impute(self, data: pd.Series) -> pd.Series:
        pass

class MeanImputation(ImputationStrategy):
    def impute(self, data: pd.Series) -> pd.Series:
        mean = data.mean()
        print(f"Điền giá trị thiếu bằng giá trị trung bình: {mean:.2f}")
        return data.fillna(mean)

class MedianImputation(ImputationStrategy):
    def impute(self, data: pd.Series) -> pd.Series:
        median = data.median()
        print(f"Điền giá trị thiếu bằng giá trị trung vị: {median:.2f}")
        return data.fillna(median)

class ConstantImputation(ImputationStrategy):
    def __init__(self, constant_value=0):
        self._constant = constant_value
    def impute(self, data: pd.Series) -> pd.Series:
        print(f"Điền giá trị thiếu bằng hằng số: {self._constant}")
        return data.fillna(self._constant)

# --- Class Context ---
class MissingValueHandler:
    def __init__(self, strategy: ImputationStrategy):
        self._strategy = strategy

    # Cho phép thay đổi chiến lược lúc runtime
    def set_strategy(self, strategy: ImputationStrategy):
        self._strategy = strategy

    def handle(self, data: pd.DataFrame) -> pd.DataFrame:
        print("\nBắt đầu xử lý giá trị thiếu...")
        data_copy = data.copy()
        for col in data_copy.columns:
            if data_copy[col].isnull().any():
                print(f"Xử lý cột '{col}':")
                # Ủy thác công việc cho object chiến lược
                data_copy[col] = self._strategy.impute(data_copy[col])
        return data_copy

# --- Sử dụng Strategy Pattern ---
data = pd.DataFrame({
    'age': [25, 30, np.nan, 45, 35],
    'salary': [50000, np.nan, 70000, 80000, 65000]
})

# 1. Sử dụng chiến lược điền bằng giá trị trung bình
mean_strategy = MeanImputation()
handler = MissingValueHandler(mean_strategy)
result1 = handler.handle(data)
print(result1)

# 2. Thay đổi chiến lược sang điền bằng hằng số -999
constant_strategy = ConstantImputation(-999)
handler.set_strategy(constant_strategy)
result2 = handler.handle(data)
print(result2)

Strategy Pattern giúp code của bạn tuân thủ **Open/Closed Principle**: class của bạn "mở" cho việc mở rộng (bạn có thể thêm chiến lược mới) nhưng "đóng" cho việc sửa đổi (bạn không cần phải sửa đổi class `MissingValueHandler`).

## 4. Singleton Pattern (Mẫu Duy nhất)

**Ý tưởng:** Đảm bảo rằng một class chỉ có **duy nhất một thể hiện (instance)** và cung cấp một điểm truy cập toàn cục đến thể hiện đó.

**Vấn đề cần giải quyết:** Khi bạn cần chính xác một đối tượng để điều phối các hành động trong toàn bộ hệ thống. Ví dụ: đối tượng quản lý cấu hình (configuration manager), đối tượng quản lý kết nối đến cơ sở dữ liệu (database connection pool), hoặc đối tượng ghi log (logger).

**Cảnh báo:** Singleton thường bị lạm dụng. Hãy cân nhắc kỹ trước khi sử dụng. Thường thì việc truyền một đối tượng dùng chung qua các hàm/constructor (dependency injection) sẽ linh hoạt và dễ test hơn.

**Ví dụ trong Data Science:** Một class quản lý cấu hình cho toàn bộ pipeline (ví dụ: đường dẫn, các siêu tham số).

In [ ]:
class PipelineConfig:
    _instance = None

    # Ghi đè __new__ để kiểm soát quá trình tạo object
    def __new__(cls, *args, **kwargs):
        if not cls._instance:
            print("Tạo instance mới cho PipelineConfig...")
            cls._instance = super(PipelineConfig, cls).__new__(cls, *args, **kwargs)
        else:
            print("Sử dụng lại instance đã có của PipelineConfig...")
        return cls._instance

    def __init__(self):
        # __init__ vẫn được gọi mỗi lần, nên cần kiểm tra để không load lại config
        if not hasattr(self, 'is_initialized'):
            print("Khởi tạo và load config...")
            self.config = {
                'data_path': '/path/to/data.csv',
                'model_name': 'RandomForest',
                'learning_rate': 0.01
            }
            self.is_initialized = True

    def get(self, key):
        return self.config.get(key)

# --- Sử dụng Singleton ---
config1 = PipelineConfig()
config2 = PipelineConfig()

# Kiểm tra xem hai biến có trỏ đến cùng một object không
print(f"\nconfig1 và config2 có phải là một không? {config1 is config2}")

# Truy cập config từ bất cứ đâu
print(f"Đường dẫn dữ liệu: {config1.get('data_path')}")
print(f"Tên mô hình: {config2.get('model_name')}")

## Tóm tắt

- **Mẫu thiết kế** là các giải pháp đã được kiểm chứng cho các vấn đề lập trình phổ biến.
- **Factory Pattern:** Sử dụng một "nhà máy" để tạo các đối tượng, giúp che giấu logic khởi tạo phức tạp và làm cho code dễ dàng mở rộng khi có thêm loại đối tượng mới.
- **Strategy Pattern:** Đóng gói các thuật toán/hành vi vào các class riêng biệt và hoán đổi chúng, giúp loại bỏ các câu lệnh `if/else` dài và tuân thủ Open/Closed Principle.
- **Singleton Pattern:** Đảm bảo một class chỉ có một instance duy nhất, hữu ích cho việc quản lý các tài nguyên toàn cục như cấu hình hoặc kết nối CSDL.
- Việc áp dụng các mẫu thiết kế phù hợp sẽ giúp bạn viết code OOP có cấu trúc, linh hoạt và chuyên nghiệp hơn, sẵn sàng cho các dự án data science phức tạp.

## Bài học tiếp theo

Chúng ta đã đi qua các khái niệm và mẫu thiết kế quan trọng nhất của OOP. Trong bài học cuối cùng, chúng ta sẽ tổng hợp tất cả kiến thức đã học để xây dựng một pipeline khoa học dữ liệu hoàn chỉnh từ đầu đến cuối bằng cách sử dụng tư duy hướng đối tượng.